
# Notebook 2 — Le facteur de réduction et les points de basculement

**Course notebook** using `DICE.py`.  
Nous travaillons sur deux blocs des diapositives : **=Le facteur de réduction** et **==Tipping points.

## Résumé du cahier
- **0)** Load `DICE.py` et lancez une ligne de base.
- **1)** **La controverse du facteur d'actualisation**
  1-A) Du facteur d'actualisation au taux d'intérêt (et exercice de codage rapide).
  1-B) Taux d'intérêt réels; choisir 3 valeurs pour`p.rho` (including Stern’s low value).  
  1-C) Calculer la politique optimale pour les 3 scénarios.
  1-D) Placer et comparer **abatement**, **taxe carbone**, **dommages**, **température**.
  1-E) Short comments.
- **2)** **Le rôle des points de basculement**
  2-A) Using `p.user_damage_fn` définir les dommages sur mesure (base Nordhaus; double coefficient).
  2-B) Résultats du lot.
  2-C) Interpret.  
  2-D) Mettre en œuvre une fonction de dommages **Weitzman** et résoudre.
  2-E) Comparer les chiffres.
  2-F) Comment.  
  2-G) Ajouter un **kink** à 3°C (les dommages doublent au-dessus du seuil) et interpréter.



## 0) Load `DICE.py`
Make sure `DICE.py` se trouve à côté de ce carnet (ou ajuster le chemin d'importation en conséquence).


In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

# Baseline initialization
p       = Params()
sim     = init_states(p)
timevec = range(1, p.nT)
sim     = update_path(sim, timevec, p)

# Peek at the dataframe if you want
df = mat_to_df(sim, p)
df.head()



## 1) La controverse sur le facteur d'actualisation

**Context.**  
Le facteur d'actualisation détermine comment nous valorisons l'avenir. Une plus grande préférence pour le temps pur$\rho$ c'est-à-dire que l'on escompte davantage l'avenir, ce qui implique généralement une réduction à court terme ** plus faible**. Moins$\rho$ (comme le préconise **Stern**) implique **fort** et **début** d'atténuation.

- **Nordhaus** (e.g., DICE): often uses $\rho \approx 1.5\%$ par an (plus la croissance et les taux de courbure dans l'équation de consommation Euler).
- **Stern (2006, Stern Review)** : plaide en faveur d'un taux de préférence pour le temps (près de zéro**) ($\rho \approx 0.1\%$ per year).  
  Lien vers la revue **Stern**:https://webarchive.nationalarchives.gov.uk/ukgwa/20100407172811/http://www.hm-treasury.gov.uk/stern_review_report.htm

This debate is ethical as much as technical: how much should current generations **value future generations' welfare**?



### 1-A) Du facteur d'actualisation au taux d'intérêt

Let $\beta = \dfrac{1}{1+\rho}$ indiquer le facteur d'actualisation du planificateur. Selon les hypothèses standard, le taux d'intérêt réel (approximatif) est
$$
r \approx \rho + \gamma g,
$$
where $g$ est la croissance de la consommation par habitant et$\gamma$ est le paramètre CRA (inverse de l'ISE).

** Exercice.**
1. Lire les paramètres actuels de réduction de`p`: `p.rho` et`p.gamma`.
2. Supposons un taux de croissance plausible de la consommation, par exemple$g=2\%$.
3. Calculer et imprimer$\beta$ et$r$. (Se sentir libre d'expérimenter avec d'autres valeurs de$g$.)


In [ ]:
rho = p.rho
gamma = p.gamma
g = 0.02

# Compute both requested quantities:
# beta = 1 / (1 + rho)
# r = rho + gamma * g



### 1-B) Real-world interest rates & choosing three $\rho$ values

Les taux d'intérêt à long terme **réels** sont généralement de 1 à 4 % par an (variant par pays et par période). On peut voir :
- US data on long term government bonds https://fred.stlouisfed.org/series/IRLTLT01USA156N
- Or a global R https://cepr.org/voxeu/columns/global-r

Pour cet exercice, considérer trois valeurs pour le taux de préférence pour le temps pur:
- **Nordhaus-like**: $\rho = 0.015$ (1.5%)
- **Intermediate**: $\rho = 0.005$ (0.5%)
- **Stern-like**: $\rho = 0.001$ (0.1%) — often summarized as $\beta \approx 0.999$.

Nous allons résoudre la politique optimale pour chacun. Définir ici simplement le correspondant$\rho$.

Tout d'abord, changer le calibrage du rho pour les autres sceanrios.


In [ ]:
# >>> Your code here <<<
pNordhaus     = Params()
pIntermediate = Params()
pStern        = Params()
# a twist to get calculation manageable, otherwise the planner windows expands a lot
pIntermediate.toly   = 0.01
pStern.toly          = 0.01
# ...


### 1-C) Calculer la politique optimale pour les trois scénarios

Nous optimisons sur **un contrôle** pour réduire le temps de calcul: le taux de réduction$\mu_t$.


In [ ]:
# >>> Your code here <<<
bounds_s   = [(0, 1)]
control_id = [pNordhaus.i_mu]
path_opt_Nordhaus = run_optimal_policy(sim.copy(), timevec, pNordhaus, bounds_s, control_id)
# ...


### 1-D) Terrain: réduction, taxe sur le carbone, dommages, températures

** Exercice.** Créer des placettes de comparaison dans les trois scénarios d'actualisation.


In [ ]:
# >>> Your code here <<<

plt.subplot(1, 4, 1)
# plot abatement
# var name: "mu"

plt.subplot(1, 4, 2)
# plot damages
# damages = p.a2 * (sim[0:,p.i_T_AT] ** p.a3)

plt.subplot(1, 4, 3)
# plot carbon tax
# var name: "Tax"

plt.subplot(1, 4, 4)
# plot temperatures
# var name: "T_AT"

plt.show()



### 1-E) Comment (short paragraph)

** Exercice.** Rédigez un court paragraphe :
- How does lowering $\rho$ modifier les voies de réduction et les impôts?
- Qu'advient-il des températures et des dommages?
- Vos résultats correspondent-ils à l'intuition des diapositives?


> ✍️ You written answer here.


## 2) Le rôle des points de basculement

**Slides recap.**  
Les points de basculement font référence à des changements brusques et potentiellement irréversibles dans le système climatique (p. ex., effondrement de la nappe glaciaire, changements de l'AMOC, dégel du pergélisol). Ils impliquent des risques **non linéaires** et **à queue grasse**. Des dommages quadratiques lisses peuvent sous-estimer les résultats extrêmes.



### 2-A) Doubler les dommages quadratiques de base

Le module DICE représente les dommages grâce aux paramètres`Params`. Les
la spécification de base est

$$D(T)=a_2T^{a_3}.$$

Create `pB = Params()` pour le niveau de référence et`pN = Params()` pour l'alternative,
then set `pN.a2 = 2 * pN.a2`. Initialiser et optimiser les deux chemins
abatement control `i_mu` utilisant le même horizon et les mêmes limites.


In [ ]:
pB = Params()
pN = Params()
pN.a2 = 2 * pN.a2

timevec = range(1, p.nT)
simB = init_states(pB)
simN = init_states(pN)
bounds_mu = (0, 1)
control_id = [pB.i_mu]

# Optimise simB with pB and simN with pN using run_optimal_policy.



### 2-B) Résultats du lot (dommages, impôt, abattement)

**Exercice.** Comparer **base** vs **double** dommages-intérêts sur dommages-intérêts, impôts et abattement (en supposant la mise en œuvre d'une abattement optimal).


In [ ]:
# Compare simB (baseline) with simN (doubled a2) in four panels.
# Use columns i_mu, i_Tax and i_T_AT.
# Compute damage shares as p.a2 * temperature ** p.a3 for each calibration.



### 2-C) Interpret (short paragraph)

** Exercice.** Expliquez comment les dommages plus élevés affectent la politique optimale :
- Comment les impôts et les abattements changent-ils au fil du temps?
- Comment la trajectoire de température réagit-elle?


> ✍️ You written answer here.

### 2-D) Weitzman-style damages (fat tails)

Utiliser le deuxième terme de dommage déjà mis en œuvre dans`DICE.py`:

$$D_W(T)=a_2T^{a_3}+a_4T^{a_5}.$$

Pour le calibrage de Weitzman, set:

```python
pW.a4 = 5.0703e-06
pW.a5 = 6.754
pW.a6 = 0.0
```

Here `a6=0` active le terme supplémentaire pour les températures positives. Simulez
le chemin optimal et, pour comparaison, un laissez-faire sans optimisation.


In [ ]:
pW = Params()
pW.a4 = 5.0703e-06
pW.a5 = 6.754
pW.a6 = 0.0

# Initialise an optimal Weitzman path and a laissez-faire copy.
# Optimise the first with run_optimal_policy and propagate the second with update_path.


### 2-E) Comparer les résultats de Nordhaus et de Weitzman

Comparer`simB`, `simW` et le laissez-faire`simW_LF`. Réduction des parcelles,
les parts de dommages, la taxe carbone et la température atmosphérique. Pour les dommages de Weitzman,
include both `a2*T**a3` et`a4*T**a5`.


In [ ]:
# Use simB, simW and simW_LF from the previous cells.
# Build four panels for i_mu, damages, i_Tax and i_T_AT.



### 2-F) Comment (short paragraph)

** Exercice.** Discutez des différences de politique inhérentes aux queues plus grosses :
- Les impôts/abattements sont-ils plus élevés et plus tôt?
- Quelle est la sensibilité des résultats à \(\kappa\) ? Essayez quelques valeurs.


> ✍️ You written answer here.

### 2-G) Introduce a kink at 3°C

Le second terme de dommages DICE est activé par seuil. Pour doubler le quadratique
damages only above 3°C, set:

```python
pG.a4 = pG.a2
pG.a5 = pG.a3
pG.a6 = 3.0
```

Optimiser ce calibrage, construire son équivalent laissez-faire, comparer les deux
avec le niveau de référence et interpréter la réponse de la réduction et la taxe sur le carbone.


In [ ]:
pG = Params()
pG.a4 = pG.a2
pG.a5 = pG.a3
pG.a6 = 3.0

# Initialise simG and simG_LF, optimise simG and propagate simG_LF.
# Then compare them with simB.


> ✍️ You written answer here.